# Process creation hunt with pandas

- Explore process creation events
- Rank parent-child pairs by rarity
- Find and decode suspicious command lines

---
<sub>TechDetechtives analytics workbench. This notebook is a modified work derived from the HELK notebook `demos/basic_event_log_analysis_pandas.ipynb` (https://github.com/Cyb3rWard0g/HELK), Copyright Roberto Rodriguez (@Cyb3rWard0g), licensed GPL-3.0. Modified by TechDetechtives, 2026: reads live process events from the platform instead of a static JSON file, uses ECS field names, and adds decoding of encoded PowerShell commands. Licensed GPL-3.0; see `analytics/LICENSE`.</sub>

In [ ]:
import base64
import re

import matplotlib.pyplot as plt
import pandas as pd
import td_hunt as td

pd.set_option("display.max_colwidth", 200)

## Load process creation events

In [ ]:
proc = td.events("process", since="24h")
proc = proc.dropna(axis=1, how="all")
proc.shape

## Look at one event

In [ ]:
proc.iloc[0]

## Group by parent process

Which programs start other programs, and how often?

In [ ]:
proc.groupby("process.parent.name").size().sort_values(ascending=False)

## Rare parent-child pairs

Pairs that occur once or twice are a better starting point than the common ones.

In [ ]:
pair_counts = (
    proc.groupby(["process.parent.name", "process.name"]).size().rename("count").reset_index()
    .sort_values("count")
)
pair_counts.head(15)

## Enrich: command line length

Very long command lines often carry encoded or inline scripts.

In [ ]:
proc["command_length"] = proc["process.command_line"].str.len()
longest = proc.sort_values("command_length", ascending=False).head(10)
longest[["host.name", "process.parent.name", "process.name", "command_length"]]

In [ ]:
ax = longest.plot(kind="barh", x="process.name", y="command_length", legend=False, figsize=(8, 4))
ax.set_xlabel("command line length (characters)")
ax.set_ylabel("")
ax.invert_yaxis()
plt.tight_layout()

## Filter on an interesting parent

The WMI provider host starts the payload of remote WMI execution, so its children deserve attention.

In [ ]:
wmi_children = proc[proc["process.parent.name"].str.lower() == "wmiprvse.exe"]
wmi_children[["@timestamp", "host.name", "user.name", "process.name", "process.command_line"]]

## Decode encoded PowerShell

PowerShell accepts a Base64 (UTF-16) script through `-EncodedCommand`. Decoding it shows what actually ran.

In [ ]:
ENCODED = re.compile(r"-e(?:nc|ncodedcommand|c)?\s+([A-Za-z0-9+/=]{16,})", re.IGNORECASE)

def decode_powershell(command_line):
    match = ENCODED.search(command_line or "")
    if not match:
        return None
    try:
        return base64.b64decode(match.group(1)).decode("utf-16-le")
    except (ValueError, UnicodeDecodeError):
        return None

proc["decoded_command"] = proc["process.command_line"].apply(decode_powershell)
proc[proc["decoded_command"].notna()][["@timestamp", "host.name", "user.name", "decoded_command"]]

## What did that process start next?

Follow the process identifier of each decoded PowerShell to its children.

In [ ]:
suspect_ids = proc.loc[proc["decoded_command"].notna(), "process.entity_id"]
children = proc[proc["process.parent.entity_id"].isin(suspect_ids)]
children[["@timestamp", "host.name", "user.name", "process.name", "process.command_line"]]